In [ ]:
!wget --no-check-certificate \
"https://drive.usercontent.google.com/download?id=1bbyqVCKZX5Ur5Zg-uKj0jD0maWAVeOLx&export=download&confirm=t" \
-O CarDD_release.zip

--2026-09-09 13:43:39--  https://drive.usercontent.google.com/download?id=1bbyqVCKZX5Ur5Zg-uKj0jD0maWAVeOLx&export=download&confirm=t
Resolving drive.usercontent.google.com (drive.usercontent.google.com)... 172.253.153.132, 2a00:1450:4013:c22::84
Connecting to drive.usercontent.google.com (drive.usercontent.google.com)|172.253.153.132|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 6045392762 (5.6G) [application/octet-stream]
Saving to: ‘CarDD_release.zip’

CarDD_release.zip   100%[===================>]   5.63G  52.0MB/s    in 90s     

2026-09-09 13:45:10 (63.9 MB/s) - ‘CarDD_release.zip’ saved [6045392762/6045392762]



In [ ]:
!unzip -q CarDD_release.zip -d CarDD_release

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!cp -r CarDD_release "/content/drive/MyDrive/"

In [ ]:
!pip install -q fiftyone opencv-python-headless

import fiftyone as fo
import json, os, random, shutil
from pathlib import Path
import numpy as np
import cv2

CLASSES = ["dent", "scratch", "crack", "glass shatter", "lamp broken", "tire flat"]
CLASS_TO_ID = {c: i for i, c in enumerate(CLASSES)}
MIN_POLYGON_POINTS = 3

def mask_to_normalized_polygon(mask, bbox_norm, img_w, img_h, epsilon_frac=0.002):
    bx, by, bw, bh = bbox_norm
    x0_px = bx * img_w
    y0_px = by * img_h

    mask_u8 = (mask.astype(np.uint8)) * 255
    contours, _ = cv2.findContours(mask_u8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None
    contour = max(contours, key=cv2.contourArea)
    if cv2.contourArea(contour) < 1:
        return None

    peri = cv2.arcLength(contour, True)
    epsilon = max(epsilon_frac * peri, 0.5)
    contour = cv2.approxPolyDP(contour, epsilon, True)

    pts = contour.reshape(-1, 2).astype(np.float64)
    if len(pts) < MIN_POLYGON_POINTS:
        return None

    pts[:, 0] = (pts[:, 0] + x0_px) / img_w
    pts[:, 1] = (pts[:, 1] + y0_px) / img_h
    pts = np.clip(pts, 0.0, 1.0)
    return pts.flatten().tolist()


SAMPLES_JSON = "/content/CarDD_hf/samples.json"
IMAGES_DIR = "/content/CarDD_hf/data"
OUT_DIR = Path("/content/cardd_yolo_seg")

for split in ("train", "val", "test"):
    (OUT_DIR / "images" / split).mkdir(parents=True, exist_ok=True)
    (OUT_DIR / "labels" / split).mkdir(parents=True, exist_ok=True)

with open(SAMPLES_JSON, "r", encoding="utf-8") as f:
    data = json.load(f)
samples = data["samples"]

random.seed(42)
indices = list(range(len(samples)))
random.shuffle(indices)
n = len(indices)
n_train = int(n * 0.8)
n_val = int(n * 0.1)
split_map = {}
for i, idx in enumerate(indices):
    split_map[idx] = "train" if i < n_train else ("val" if i < n_train + n_val else "test")

stats = {"images_ok": 0, "images_skipped": 0, "instances_ok": 0, "instances_skipped": 0}
class_counts = {c: 0 for c in CLASSES}

for idx, sample in enumerate(samples):
    split = split_map[idx]
    img_w = sample["metadata"]["width"]
    img_h = sample["metadata"]["height"]

    img_name = os.path.basename(sample["filepath"])
    src_img_path = Path(IMAGES_DIR) / img_name
    if not src_img_path.exists():
        stats["images_skipped"] += 1
        continue

    detections = sample.get("segmentations", {}).get("detections", [])
    if not detections:
        stats["images_skipped"] += 1
        continue

    lines = []
    for det_dict in detections:
        label = det_dict.get("label")
        if label not in CLASS_TO_ID:
            continue

        det = fo.Detection.from_dict(det_dict)
        if not det.has_mask:
            stats["instances_skipped"] += 1
            continue

        poly = mask_to_normalized_polygon(det.mask, det.bounding_box, img_w, img_h)
        if poly is None:
            stats["instances_skipped"] += 1
            continue

        class_id = CLASS_TO_ID[label]
        coords_str = " ".join(f"{v:.6f}" for v in poly)
        lines.append(f"{class_id} {coords_str}")
        class_counts[label] += 1
        stats["instances_ok"] += 1

    if not lines:
        stats["images_skipped"] += 1
        continue

    stem = Path(img_name).stem
    dst_img = OUT_DIR / "images" / split / img_name
    dst_lbl = OUT_DIR / "labels" / split / f"{stem}.txt"
    shutil.copy2(src_img_path, dst_img)
    with open(dst_lbl, "w") as f:
        f.write("\n".join(lines) + "\n")
    stats["images_ok"] += 1

yaml_content = (
    f"path: {OUT_DIR.resolve()}\n"
    f"train: images/train\n"
    f"val: images/val\n"
    f"test: images/test\n"
    f"names:\n" + "\n".join(f"  {i}: {c}" for i, c in enumerate(CLASSES)) + "\n"
)
with open(OUT_DIR / "data.yaml", "w", encoding="utf-8") as f:
    f.write(yaml_content)



-----

-----

In [ ]:
!pip install -q ultralytics pycocotools

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 50.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.4/75.4 kB 8.4 MB/s eta 0:00:00


In [ ]:
yaml_content = """
path: /content/cardd_yolo
train: images/train
val: images/val
test: images/test

names:
  0: dent
  1: scratch
  2: crack
  3: glass shatter
  4: lamp broken
  5: tire flat
"""

with open("/content/cardd_yolo/data.yaml", "w") as f:
    f.write(yaml_content)

print(open("/content/cardd_yolo/data.yaml").read())

In [ ]:
from ultralytics.data.utils import check_det_dataset
check_det_dataset("/content/cardd_yolo/data.yaml")

In [ ]:
from ultralytics import YOLO

DATA_YAML = "/content/cardd_yolo/data.yaml"

model = YOLO("yolo11m-seg.pt")

model.train(
    data=DATA_YAML,
    epochs=40,
    batch=8,
    patience=10,

    save=True,
    save_period=1,

    name="yolo11m-seg_cardd",
    project="/content/drive/MyDrive/CarDD_release/runs",
    exist_ok=True,

    # --- Optimizer & LR schedule ---
    optimizer="Adam",
    lr0=0.001,
    lrf=0.01,

    # --- Regularization ---
    weight_decay=0.0008,  # Ridge
    dropout=0.1,

    # --- Augmentation ---
    degrees=10,
    translate=0.1,
    scale=0.5,
    shear=2.0,
    perspective=0.0002,
    flipud=0.0,
    fliplr=0.5,
    hsv_h=0.015,
    hsv_s=0.6,
    hsv_v=0.4,
    mosaic=1.0,
    mixup=0.15,
    copy_paste=0.1,

    seed=42,
)


In [ ]:
!gdown https://drive.google.com/uc?id=1SLRnud67gB85SVEx4IOOqePeKYNa3YPE -O last.pt

Downloading...
From (original): https://drive.google.com/uc?id=1SLRnud67gB85SVEx4IOOqePeKYNa3YPE
From (redirected): https://drive.google.com/uc?id=1SLRnud67gB85SVEx4IOOqePeKYNa3YPE&confirm=t&uuid=b38741c8-cac3-45ec-8916-b7b125dda6d8
To: /content/last.pt
100% 180M/180M [00:03<00:00, 59.7MB/s]


In [ ]:
metrics = model.val(data=DATA_YAML, split="test")
print("Test mAP50:", metrics.seg.map50)
print("Test mAP50-95:", metrics.seg.map)
print("Test Precision:", metrics.seg.mp)
print("Test Recall:", metrics.seg.mr)

---

In [ ]:
from ultralytics.data.converter import convert_coco

DATA_ROOT = "/content/CarDD_release/CarDD_release/CarDD_COCO"

convert_coco(
    labels_dir=f"{DATA_ROOT}/annotations",
    save_dir=f"{DATA_ROOT}/yolo_labels",
    use_segments=True,
    use_keypoints=False,
    cls91to80=False
)

Annotations /content/CarDD_release/CarDD_release/CarDD_COCO/annotations/instances_test2017.json: 100% ━━━━━━━━━━━━ 374/374 1.6Kit/s 0.2s
Annotations /content/CarDD_release/CarDD_release/CarDD_COCO/annotations/instances_train2017.json: 100% ━━━━━━━━━━━━ 2816/2816 3.9Kit/s 0.7s
Annotations /content/CarDD_release/CarDD_release/CarDD_COCO/annotations/instances_val2017.json: 100% ━━━━━━━━━━━━ 810/810 3.6Kit/s 0.2s
COCO data converted successfully.
Results saved to /content/CarDD_release/CarDD_release/CarDD_COCO/yolo_labels


In [ ]:
import os, shutil

YOLO_ROOT = "/content/cardd_yolo"
os.makedirs(YOLO_ROOT, exist_ok=True)

splits_map = {"train2017": "train", "val2017": "val", "test2017": "test"}

for coco_split, yolo_split in splits_map.items():
    img_src = f"{DATA_ROOT}/{coco_split}"
    lbl_src = f"{DATA_ROOT}/yolo_labels/labels/{coco_split}"

    img_dst = f"{YOLO_ROOT}/images/{yolo_split}"
    lbl_dst = f"{YOLO_ROOT}/labels/{yolo_split}"

    os.makedirs(img_dst, exist_ok=True)
    os.makedirs(lbl_dst, exist_ok=True)

    if os.path.exists(img_src):
        for f in os.listdir(img_src):
            shutil.copy(os.path.join(img_src, f), img_dst)

    if os.path.exists(lbl_src):
        for f in os.listdir(lbl_src):
            shutil.copy(os.path.join(lbl_src, f), lbl_dst)

    print(f"{yolo_split}: {len(os.listdir(img_dst))} images, {len(os.listdir(lbl_dst))} labels")

train: 2816 images, 2816 labels
val: 810 images, 810 labels
test: 374 images, 374 labels


In [ ]:
!ls /content/cardd_yolo/images/train | sed 's/\.[^.]*$//' | sort > /tmp/img_names.txt
!ls /content/cardd_yolo/labels/train | sed 's/\.[^.]*$//' | sort > /tmp/lbl_names.txt
!echo "Images:"; wc -l /tmp/img_names.txt
!echo "Labels:"; wc -l /tmp/lbl_names.txt
!echo "Matching:"; comm -12 /tmp/img_names.txt /tmp/lbl_names.txt | wc -l

Images:
2816 /tmp/img_names.txt
Labels:
2816 /tmp/lbl_names.txt
Matching:
2816


In [ ]:
yaml_content = """
path: /content/cardd_yolo
train: images/train
val: images/val
test: images/test

names:
  0: dent
  1: scratch
  2: crack
  3: glass shatter
  4: lamp broken
  5: tire flat
"""
with open("/content/cardd_yolo/data.yaml", "w") as f:
    f.write(yaml_content)

In [ ]:
!gdown https://drive.google.com/uc?id=1SLRnud67gB85SVEx4IOOqePeKYNa3YPE -O last.pt

from ultralytics import YOLO
model = YOLO("last.pt")

model.train(
    data="/content/cardd_yolo/data.yaml",
    resume=True,
    project="/content/runs",
    name="cardd_continue",
)

Downloading...
From (original): https://drive.google.com/uc?id=1SLRnud67gB85SVEx4IOOqePeKYNa3YPE
From (redirected): https://drive.google.com/uc?id=1SLRnud67gB85SVEx4IOOqePeKYNa3YPE&confirm=t&uuid=09f77080-42ae-4b9d-b67a-0145a720ce38
To: /content/last.pt
100% 180M/180M [00:02<00:00, 66.5MB/s]
Ultralytics 8.4.145 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=8, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/cardd_yolo/data.yaml, degrees=10, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.1, dynamic=False, embed=None, epochs=40, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freez

ultralytics.utils.metrics.SegmentMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4, 5])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7a8d641e2120>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)', 'Precision-Recall(M)', 'F1-Confidence(M)', 'Precision-Confidence(M)', 'Recall-Confidence(M)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004

In [ ]:
from google.colab import files

files.download("/content/drive/MyDrive/CarDD_release/runs/yolo11m-seg_cardd/weights/last.pt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
files.download("/content/drive/MyDrive/CarDD_release/runs/yolo11m-seg_cardd/weights/best.pt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>